# kazakh-parenting-llm — full pipeline on Kaggle

**Settings → Accelerator: GPU T4 x2, Internet: On.** Total runtime ≈ 1.5–2 h (most of it training).

Optional **Add-ons → Secrets**:
- `TEACHER_API_KEY` — free Gemini API key (aistudio.google.com) for synthetic data + LLM-as-judge. Without it the notebook trains on the hand-written seed set only and skips the judge.
- `HF_TOKEN` — only if you want to push the merged model to the Hugging Face Hub.

Steps: index → data → **baseline eval** → QLoRA training → eval of 4 variants → GGUF export.

In [ ]:
!git clone -q https://github.com/AskhatAmankulov/kazakh-parenting-llm.git
%cd kazakh-parenting-llm
!pip install -q -U "transformers>=4.51" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.43" sentence-transformers rank_bm25 sacrebleu openai tabulate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("TEACHER_API_KEY", "HF_TOKEN"):
        try:
            os.environ[k] = s.get_secret(k)
            print(k, "loaded")
        except Exception:
            print(k, "not set")
except ImportError:
    print("not on Kaggle — set env vars yourself")
HAS_TEACHER = bool(os.environ.get("TEACHER_API_KEY"))
import subprocess, sys

def run(cmd):
    """Run a pipeline step, stream its output, and STOP the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if proc.wait() != 0:
        raise RuntimeError(f"STEP FAILED: {cmd}\n--- last lines ---\n" + "".join(tail))

## 1. Knowledge base → hybrid index (BM25 + bge-m3)

In [ ]:
run('python scripts/build_index.py')

## 2. Training data
Seed (72 hand-written kk/ru examples) + grounded synthetic Q&A from the teacher model (≈4 per KB chunk per round),
then cleaning: language check, length filter, exact + near-duplicate removal, eval-leakage guard.
Half of the examples are trained **with** retrieved context in the prompt, half without.

In [ ]:
if HAS_TEACHER:
    run('python scripts/generate_synthetic.py --rounds 2')
else:
    print("no TEACHER_API_KEY -> seed data only")
run('python scripts/prepare_dataset.py')

In [ ]:
import json
rows = [json.loads(l) for l in open("data/processed/train.jsonl", encoding="utf-8")]
ex = next(r for r in rows if r["lang"] == "kk" and r["origin"] != "seed") if any(r["origin"] != "seed" for r in rows) else rows[0]
for m in ex["messages"]:
    print(f"--- {m['role']} ---\n{m['content'][:700]}\n")

## 3. Baseline — base model with and without RAG (before any training)

In [ ]:
run('python scripts/generate_answers.py --variants base base_rag')

## 4. QLoRA fine-tuning (4-bit NF4 + LoRA r=16 on all linear layers, loss on answer tokens only)

In [ ]:
run('python scripts/train_qlora.py')

In [ ]:
from IPython.display import Image, display
import json
display(Image("artifacts/lora-qwen3-4b/loss.png"))
log = json.load(open("artifacts/lora-qwen3-4b/train_log.json"))
print({k: log[k] for k in ("train_minutes", "n_train", "n_val", "gpu", "peak_gpu_mem_gb")}, log["final_eval"])

## 5. Fine-tuned model, with and without RAG → evaluation of all 4 variants

In [ ]:
run('python scripts/generate_answers.py --variants ft ft_rag')
run("python scripts/evaluate.py" + (" --judge" if HAS_TEACHER else ""))

In [ ]:
from IPython.display import Markdown
display(Image("results/metrics.png"))
display(Markdown(open("results/summary.md", encoding="utf-8").read()))

### Side by side: the same Kazakh question across variants

In [ ]:
import json
qid = "ev-kk-11"   # change to any id from data/eval/eval.yaml
for v in ["base", "base_rag", "ft", "ft_rag"]:
    for l in open(f"results/generations/{v}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if r["id"] == qid:
            print(f"===== {v} =====\n{r['answer']}\n")

## 6. Export: merge LoRA → GGUF (Q4_K_M runs on a laptop CPU)
Optional, ≈10 min. Uncomment the push line to upload the merged model (private) to your HF account.

In [ ]:
run('python scripts/merge_adapter.py')
# !python scripts/merge_adapter.py --push YOUR_HF_USERNAME/qwen3-4b-kk-ru-parenting
run('bash scripts/export_gguf.sh')

## 7. Save results
Download `results.zip` from the Output panel, unzip it into the repo and commit `results/` + the adapter's
`train_log.json` / `loss.png` — the README links to them.

In [ ]:
!mkdir -p results/train && cp artifacts/lora-qwen3-4b/train_log.json artifacts/lora-qwen3-4b/loss.png data/processed/stats.json results/train/
!zip -qr /kaggle/working/results.zip results
!ls -lh /kaggle/working/results.zip